# Stratification

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
pages `examples/06-stratification-introduction`, `examples/10-derived-outputs-stratified` and
`examples/11-flows-between-strata` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Stratifying a model means splitting every compartment into groups, such as age groups or
risk groups, so that the groups can differ from one another. This page splits a small SIR
model into a young and an old group and covers:

- a `Property` (a way of classifying people, here `age`) and its `Trait`s (the groups,
  here `young` and `old`);
- applying the stratification to the model with `stratify`;
- making flows work differently in different strata with `Multiply`;
- a starting population that differs by stratum;
- adding the strata back up, or keeping them apart, with `sum_by`.

## Properties and traits

A `Property` is a named way of classifying people. `state` classifies them as susceptible,
infectious or recovered, and `age` classifies them as young or old. The values a property can
take are its *traits*, and `age["old"]` picks out one of them. Anywhere a flow or an output
needs to say "only the old people", it uses that trait.

The next cell defines the two properties and builds the unstratified SIR model. Nothing is
drawn.

In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowMass,
    FlowModel,
    Multiply,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    Split,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
age = Property("age", ("young", "old"))

model = FlowModel(PropertyMap.from_property(state))
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("foi", infectious=state["I"], contact_rate=Param("beta")),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
model.pmap.labels()

## Splitting the compartments

`model.stratify(age)` applies the stratification: every compartment is split in two, one per
age group, and the flows are repeated within each group. The list of compartments below
should now have six names, each combining a state with an age group.

To keep a stratification to some compartments only, pass `where=`. For example
`model.stratify(severity, where=state["I"])` would split only the infectious compartment.

In [ ]:
model.stratify(age)
model.pmap.labels()

## Different flows in different strata

`adjust_flow` changes a flow in some strata only. `Multiply(0.5, where=age["young"])` halves
the infection rate for the young group, and `Multiply(0.5, where=age["old"])` halves the
recovery rate for the old group, so old people stay infectious for twice as long. Anyone not
picked out by `where` is left as before.

The starting population is given by stratum, 700 young and 300 old, and a `Split` then shares
each group out between the states, with 1% of each group infectious. The cell below changes
the model and compiles it.

In [ ]:
model.adjust_flow("infection", Multiply(0.5, where=age["young"]))
model.adjust_flow("recovery", Multiply(0.5, where=age["old"]))
model.set_initial_population(
    {age["young"]: 700.0, age["old"]: 300.0},
    [Split(state, {"S": 0.99, "I": 0.01, "R": 0.0})],
)
cm = model.compile()
params = {"beta": 0.5, "gamma": 0.2}

## Adding up strata, or keeping them apart

The outputs from the previous page work here too, and `sum_by` now has a second use. Without
an argument it adds up everything. With a property, `sum_by(age)` adds up the states but
*keeps the age groups apart*, giving one column per group. You can read it as "sum, but by
age".

The cell below asks for the infectious people in three ways, then solves and evaluates the
set. The plot shows one line per age group and, dashed, the total. The two groups
peak at about the same time, near day 27, and the dashed line is the sum of the other two.
Both peaks are about 90 people, even though there are only 300 old people against 700 young,
because the old recover more slowly.

In [ ]:
outputs = OutputSet()
outputs["infectious"] = Compartments(where=state["I"])
outputs["infectious_by_age"] = outputs.ref("infectious").sum_by(age)
outputs["infectious_total"] = outputs.ref("infectious").sum_by()

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

by_age = named["infectious_by_age"].to_pandas()
by_age["total"] = named["infectious_total"].to_pandas()["value"]
fig = by_age.plot(
    title="Infectious people by age group",
    labels={"time": "day", "value": "people", "variable": ""},
)
fig.data[-1].update(line={"dash": "dash"})
fig

## Prevalence in each group

Outputs with age groups can be divided by one another, group by group. Dividing the
infectious people in each age group by the number of people in that group gives the
prevalence within each group. The plot shows the old group (slower recovery) reaching a
higher prevalence than the young group (lower infection rate), in line with how the
adjustments were set.

In [ ]:
outputs["population_by_age"] = Compartments().sum_by(age)
outputs["prevalence_by_age"] = outputs.ref("infectious_by_age") / outputs.ref("population_by_age")

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

(100 * named["prevalence_by_age"]).to_pandas().plot(
    title="Prevalence within each age group",
    labels={"time": "day", "value": "% of the age group", "variable": ""},
)

## Flows by stratum

Flows can be split by stratum too. An infection moves people from S to I, and the group of
the person is the same at both ends, so `FlowMass("infection").sum_by(age, side="dest")`
gives the rate of new infections in each age group. `side` says whether to group by the
compartment the flow leaves (`"source"`) or the one it arrives in (`"dest"`); it is needed
whenever a flow is grouped. `cumulative_integral()` then counts the people infected so far in
each group. The plot should show the running totals levelling off at the number of people who
were ever infected in each group.

In [ ]:
outputs["incidence_by_age"] = FlowMass("infection").sum_by(age, side="dest")
outputs["infected_so_far"] = outputs.ref("incidence_by_age").cumulative_integral()

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

named["infected_so_far"].to_pandas().plot(
    title="People infected so far, by age group",
    labels={"time": "day", "value": "people", "variable": ""},
)

## Splitting a flow by a parameter

So far each flow kept people in their own stratum. A flow can also send them into strata that
exist only at its destination. The next model has three stages. Only the active stage is split by
`focus`, so a person moving from incipient to active must land in one of the focus strata. By
default they are shared equally. `split=` gives the proportions instead, and the proportions can
be parameters: here `prop_infectious` of the people progressing become high-focus.

A `split=` maps each destination-only property to one proportion per trait. A proportion can be
a number or any rate expression: a `Param`, arithmetic on parameters such as `1 - p`, or a
function of time. Numbers must add up to 1. Expressions can only be checked when the model runs,
so they are used as relative weights: summer4 divides each one by their total. A trait that
should get nobody is still named, with `0.0`.

The clearance flow shows the other tool from earlier in this page, an adjustment.
`Multiply(Param("high_focus_clearance"), where=focus["high"])` scales clearance in the
high-focus stratum by a parameter.

In [ ]:
from summer4 import Dest

stage = Property("stage", ("incipient", "active", "cleared"))
focus = Property("focus", ("low", "high"))
tb = FlowModel(PropertyMap.from_property(stage).stratify(focus, where=stage["active"]))

p = Param("prop_infectious")
tb.add_flow(
    TransitionFlow(
        "progression",
        stage["incipient"],
        stage["active"],
        Param("progression_rate"),
        split={focus: {"low": 1 - p, "high": p}},
    )
)
tb.add_flow(
    TransitionFlow(
        "clearance",
        stage["active"],
        stage["cleared"],
        Param("clearance_rate"),
        adjust=[Multiply(Param("high_focus_clearance"), where=focus["high"])],
    )
)
tb.set_initial_population({stage["incipient"]: 1000.0})
tb_cm = tb.compile()
tb_params = {
    "progression_rate": 0.1,
    "prop_infectious": 0.3,
    "clearance_rate": 0.2,
    "high_focus_clearance": 0.5,
}

The plot shows the progression flow by destination stratum. At every time, the high-focus
line is 30% of the total, because `prop_infectious` is 0.3. Change `prop_infectious` in
`tb_params` and run the cell again: the split moves, and the model is not rebuilt.

In [ ]:
tb_outputs = OutputSet()
tb_outputs["progression_by_focus"] = FlowMass("progression").sum_by(focus, side="dest")
tb_outputs["active_by_focus"] = Compartments(where=stage["active"]).sum_by(focus)

tb_result = tb_cm.run(tb_params, t0=0.0, t1=60.0, dt=1.0, save=tb_outputs.plan())
tb_named = tb_outputs.evaluate(tb_result, tb_params)
tb_named["progression_by_focus"].to_pandas().plot(
    title="Progression to active, by focus",
    labels={"time": "day", "value": "people per day", "variable": ""},
)

## What an adjustment can be

`adjust=` on a flow (or `adjust_flow(name, ...)` afterwards) takes a list. Each item is one of
three kinds:

| Item | Effect on the flow's rate |
|---|---|
| `Multiply(value, where=None)` | multiplies it by `value` |
| `Overwrite(value, where=None)` | replaces it with `value` |
| `Transform(fn, *args, where=None)` | replaces it with `fn(rate, *args)` |

A bare number or rate expression in the list means `Multiply(value)` with no `where`. `value`
and `args` can be numbers, parameters or any rate expression. `where` picks the strata the item
applies to, and everything else is left alone. On a flow between two strata a bare selector
looks at the source. Wrap it in `Dest(...)` or `Source(...)` to choose the end.

The order in the list does not decide what happens first. Every `Overwrite` is applied first,
then every `Multiply`, then every `Transform`, and items of the same kind keep their list
order. Pass `precedence=` to an item to change its place.

`Dest(...)` lets one flow treat its destinations differently. The cell below uses it to write
the same parameterised split as adjustments instead. The flow's default split is equal,
one half each, so the factors are doubled. The dashed lines should lie on the solid ones.
`split=` says the same thing more directly; adjustments are the tool when the rate itself
changes by stratum.

In [ ]:
tb_adjusted = FlowModel(tb.pmap)
tb_adjusted.add_flow(
    TransitionFlow(
        "progression",
        stage["incipient"],
        stage["active"],
        Param("progression_rate"),
        adjust=[
            Multiply(2 * (1 - p), where=Dest(focus["low"])),
            Multiply(2 * p, where=Dest(focus["high"])),
        ],
    )
)
tb_adjusted.add_flow(tb.flows[1])
tb_adjusted.set_initial_population({stage["incipient"]: 1000.0})

adjusted_result = tb_adjusted.compile().run(
    tb_params, t0=0.0, t1=60.0, dt=1.0, save=tb_outputs.plan()
)
adjusted = tb_outputs.evaluate(adjusted_result, tb_params)["active_by_focus"].to_pandas()
both = tb_named["active_by_focus"].to_pandas().add_suffix(" (split=)")
both = both.join(adjusted.add_suffix(" (adjust=)"))
fig = both.plot(
    title="Active people by focus: split= and adjust= agree",
    labels={"time": "day", "value": "people", "variable": ""},
)
for trace in fig.data[2:]:
    trace.update(line={"dash": "dash"})
fig

## Things to try

- Change the halving factors in the two `Multiply` calls, or swap `young` and `old`. Which
  group has the larger epidemic?
- Add a second property, such as `risk = Property("risk", ("low", "high"))`, and stratify by it
  as well. Then compare `sum_by(age)`, `sum_by(risk)` and `sum_by(age, risk)`. How many
  columns does each give?
- Make the starting population 300 young and 700 old. How does the total epidemic change?
- Plot `FlowMass("recovery").sum_by(age, side="source")` to see recoveries by age group.
- In the focus model, make `prop_infectious` a function of time, such as
  `Time() / 60` (`from summer4 import Time`), so the high-focus share grows during the run.
